# 7-Eleven AI Functions Demo

This notebook showcases **Databricks AI Functions** using the 7-Eleven demo data.

**Prerequisites:**
1. Run **01_setup_7eleven_demo_data** to create tables in `${catalog_name}.${schema_name}`.
2. Set `catalog_name` and `schema_name` in the next cell to match.
3. Use a cluster with **Databricks Runtime 15.1+** in a region that supports [AI Functions](https://docs.databricks.com/en/large-language-models/ai-functions.html).

**Functions demonstrated:**
| Function | Use case |
|----------|----------|
| **ai_query** | Natural language over structured data (sales, stores) |
| **ai_summarize** | Short summaries of long customer reviews |
| **ai_analyze_sentiment** | Sentiment (positive/negative/neutral/mixed) on review text |
| **ai_gen** | Generate product copy or category descriptions |
| **ai_similarity** | Find products or reviews similar to a reference |
| **ai_extract** | Extract entities (products, issues, locations) from text |
| **ai_classify** | Classify feedback by topic or priority |

In [0]:
-- Configuration: set your catalog and schema
DECLARE OR REPLACE VARIABLE catalog_name STRING DEFAULT 'renjiharold_demo';
DECLARE OR REPLACE VARIABLE schema_name STRING DEFAULT 'ai_functions_demo';

USE IDENTIFIER(catalog_name || '.' || schema_name);

SELECT 'Schema ready: ' || current_schema() AS status;

---
## 1. ai_query – Natural language over your data

Use a foundation model to answer questions in plain English. We build context from our tables and pass it to the model.

**7-Eleven use case:** "Which categories drive the most revenue?" or "Summarize sales by region."

In [0]:
-- Build a short text summary of sales by category (context for the model)
CREATE OR REPLACE TEMP VIEW sales_context AS
SELECT
  CONCAT(
    'Sales data: Category ', category,
    ' had total revenue ', CAST(ROUND(SUM(revenue), 0) AS STRING),
    ' and ', CAST(SUM(units_sold) AS STRING), ' units sold.'
  ) AS line
FROM renjiharold_demo.ai_functions_demo.daily_sales
GROUP BY category;

SELECT COLLECT_LIST(line) AS context FROM (
  SELECT line FROM sales_context
);

In [0]:
-- ai_query: ask a question in natural language (uses Foundation Model API endpoint)
-- Replace the endpoint name if your workspace uses a different foundation model.
-- First create a single-row table with the full context, then call ai_query.
WITH context_str AS (
  SELECT CONCAT_WS(' ', COLLECT_LIST(line)) AS ctx FROM sales_context
)
SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Based on this sales data, which product category had the lowest total revenue? Answer in one short sentence. Data: ' || ctx
) AS answer
FROM context_str;

---
## 2. ai_summarize – Short summaries of long text

**7-Eleven use case:** Summarize long customer reviews for dashboards or reports (e.g. 50-word cap).

In [0]:
SELECT
  review_id,
  store_id,
  LEFT(review_text, 60) || '...' AS original_preview,
  ai_summarize(review_text, 25) AS summary_25_words
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.customer_reviews')
LIMIT 5;

---
## 3. ai_analyze_sentiment – Positive / negative / neutral / mixed

**7-Eleven use case:** Score sentiment of reviews or survey comments to track satisfaction and flag issues.

In [0]:
SELECT
  review_id,
  store_id,
  rating,
  ai_analyze_sentiment(review_text) AS sentiment,
  LEFT(review_text, 70) || '...' AS review_preview
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.customer_reviews')
ORDER BY review_id;

---
## 4. ai_gen – Generate text from a prompt

**7-Eleven use case:** Generate product descriptions, promo copy, or category blurbs from product/category names.

In [0]:
SELECT
  product_name,
  category,
  ai_gen('Write one short, appealing sentence to describe this convenience store product for a shelf label. Product: ' || product_name || ' (' || short_description || ').') AS generated_description
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.product_catalog')
WHERE category = 'Prepared Food'
LIMIT 4;

---
## 5. ai_similarity – Semantic similarity between two strings

**7-Eleven use case:** Find products or reviews most similar to a reference (e.g. "energy drink" → match Monster, Red Bull).

In [0]:
SELECT
  product_name,
  category,
  ROUND(ai_similarity(CONCAT(product_name, ' ', short_description), 'energy drink caffeine'), 3) AS similarity_score
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.product_catalog')
ORDER BY similarity_score DESC
LIMIT 6;

In [0]:
-- Similarity for reviews: find reviews most similar to a reference concern (e.g. cleanliness)
SELECT
  review_id,
  store_id,
  ROUND(ai_similarity(review_text, 'clean store restroom bathroom hygiene'), 3) AS cleanliness_similarity,
  LEFT(review_text, 80) || '...' AS review_preview
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.customer_reviews')
ORDER BY cleanliness_similarity DESC
LIMIT 5;

---
## 6. ai_extract – Extract entities from text

**7-Eleven use case:** Pull out product names, store issues, or locations mentioned in reviews or feedback.

In [0]:
SELECT
  review_id,
  store_id,
  review_text,
  ai_extract(review_text, ARRAY('product or item mentioned', 'main complaint or praise', 'location or area')) AS extracted
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.customer_reviews')
LIMIT 4;

---
## 7. ai_classify – Classify text into labels

**7-Eleven use case:** Classify feedback notes by topic (equipment, cleanliness, product, staff, other) or by priority.

In [0]:
SELECT
  note_id,
  store_id,
  LEFT(note_text, 80) || '...' AS note_preview,
  ai_classify(note_text, ARRAY('Equipment or machine issue', 'Cleanliness or facility', 'Product quality or availability', 'Staff or service', 'Positive feedback', 'Other')) AS feedback_topic
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.feedback_notes')
ORDER BY note_id;

In [0]:
-- Classify customer reviews by primary concern (for routing or reporting)
SELECT
  review_id,
  rating,
  ai_classify(review_text, ARRAY('Food quality', 'Store cleanliness', 'Staff service', 'Product availability', 'Price or value', 'General positive', 'General negative')) AS primary_concern,
  LEFT(review_text, 50) || '...' AS preview
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.customer_reviews')
LIMIT 6;

---
## 8. Combine AI functions in one query

Example: sentiment + classification + short summary for a review dashboard.

In [0]:
SELECT
  review_id,
  store_id,
  rating,
  ai_analyze_sentiment(review_text) AS sentiment,
  ai_classify(review_text, ARRAY('Food', 'Cleanliness', 'Staff', 'Product availability', 'Other')) AS topic,
  ai_summarize(review_text, 15) AS short_summary
FROM IDENTIFIER(catalog_name || '.' || schema_name || '.customer_reviews')
LIMIT 5;